# PGFCL: Prototype-Guided Federated Contrastive Learning
### Full Experimental Notebook - Elliptic + IBM AML Datasets

In [ ]:
import torch
import os

# Detect torch version for correct PyG wheels
TORCH = torch.__version__.split('+')[0]   # e.g. '2.3.0'
CUDA  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH}  cuda={CUDA}')

# Install PyG and sparse dependencies
os.system(f'pip install torch-geometric -q')
os.system(
    f'pip install torch-scatter torch-sparse '
    f'--find-links https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q'
)

# Verify
import importlib
for pkg in ['torch_geometric', 'torch_scatter', 'torch_sparse']:
    try:
        importlib.import_module(pkg)
        print(f'  OK: {pkg}')
    except ImportError as e:
        print(f'  MISSING: {pkg} -- {e}')


torch=2.10.0  cuda=cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 88.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 2.4 MB/s eta 0:00:00


## Section 1: Environment Setup

In [ ]:
import os, glob, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                              precision_score, recall_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HIDDEN, EMB, HEADS = 64, 32, 4
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')


## Section 2A: Elliptic Bitcoin Dataset

In [ ]:
# Your exact Kaggle path
ELLIPTIC_DIR = '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset'

# Fallback: recursive search
if not os.path.isfile(os.path.join(ELLIPTIC_DIR, 'elliptic_txs_features.csv')):
    hits = glob.glob('/kaggle/input/**/elliptic_txs_features.csv', recursive=True)
    if hits:
        ELLIPTIC_DIR = os.path.dirname(hits[0])
    else:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'elliptic_txs_features.csv' in files:
                ELLIPTIC_DIR = root; break
        else:
            raise FileNotFoundError('Cannot find elliptic_txs_features.csv')
print(f'Elliptic dir: {ELLIPTIC_DIR}')


def load_elliptic(data_dir=None):
    if data_dir is None: data_dir = ELLIPTIC_DIR
    print('Loading Elliptic dataset...')
    features = pd.read_csv(f'{data_dir}/elliptic_txs_features.csv', header=None)
    edges    = pd.read_csv(f'{data_dir}/elliptic_txs_edgelist.csv')
    classes  = pd.read_csv(f'{data_dir}/elliptic_txs_classes.csv')

    node_ids  = features.iloc[:, 0].values
    id2idx    = {nid: i for i, nid in enumerate(node_ids)}
    timesteps = features.iloc[:, 1].values.astype(int)
    raw_feats = features.iloc[:, 2:].values.astype(np.float32)

    # Per-timestep deviation features (doubles feature dim to 330)
    dev_feats = np.zeros_like(raw_feats)
    for t in np.unique(timesteps):
        mask = timesteps == t
        mu   = raw_feats[mask].mean(0)
        sd   = raw_feats[mask].std(0) + 1e-8
        dev_feats[mask] = (raw_feats[mask] - mu) / sd
    all_feats = np.concatenate([raw_feats, dev_feats], axis=1)
    all_feats = StandardScaler().fit_transform(all_feats).astype(np.float32)

    # Labels: '1'=illicit->1, '2'=licit->0, 'unknown'->-1
    classes['class'] = classes['class'].map({'1': 1, '2': 0, 'unknown': -1})
    label_map = dict(zip(classes['txId'], classes['class']))
    labels    = np.array([label_map.get(nid, -1) for nid in node_ids])

    # Edge index - validate both endpoints
    valid_edges = [(id2idx[u], id2idx[v])
                   for u, v in zip(edges.iloc[:, 0], edges.iloc[:, 1])
                   if u in id2idx and v in id2idx]
    n_dropped = len(edges) - len(valid_edges)
    if n_dropped: print(f'  Warning: dropped {n_dropped} edges.')
    srcs, dsts = zip(*valid_edges) if valid_edges else ([], [])
    edge_index = torch.tensor([list(srcs), list(dsts)], dtype=torch.long)

    data = Data(
        x          = torch.tensor(all_feats),
        edge_index = edge_index,
        y          = torch.tensor(labels, dtype=torch.long),
        timestep   = torch.tensor(timesteps, dtype=torch.long)
    )
    print(f'  Nodes: {data.num_nodes} | Edges: {data.num_edges} | Features: {data.num_node_features}')
    print(f'  Illicit: {(labels==1).sum()} | Licit: {(labels==0).sum()} | Unknown: {(labels==-1).sum()}')
    return data

elliptic_data = load_elliptic()


## Section 2B: IBM AML Dataset

In [ ]:
IBM_BASE = '/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml'
IBM_FILENAMES = ['HI-Small_Trans.csv','HI-Medium_Trans.csv','HI-Large_Trans.csv',
                 'LI-Small_Trans.csv','LI-Medium_Trans.csv','LI-Large_Trans.csv']

IBM_PATH = None
for fname in IBM_FILENAMES:
    p = os.path.join(IBM_BASE, fname)
    if os.path.isfile(p): IBM_PATH = p; break
if IBM_PATH is None:
    for fname in IBM_FILENAMES:
        hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if hits: IBM_PATH = hits[0]; break
if IBM_PATH is None:
    raise FileNotFoundError('Cannot find IBM AML CSV. Add dataset ealtman2019/ibm-transactions-for-anti-money-laundering-aml')
print(f'IBM file: {IBM_PATH}')


def load_ibm_aml(filepath=None, max_nodes=80_000):
    if filepath is None: filepath = IBM_PATH
    print('Loading IBM AML dataset...')
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    print(f'  Raw transactions: {len(df)} | Columns: {list(df.columns[:8])}')

    # Timestamp -> 49 temporal bins
    ts_col = next((c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()), None)
    ref_col = ts_col if ts_col else ('Timestamp' if 'Timestamp' in df.columns else None)
    if ref_col:
        df['_ts'] = pd.to_datetime(df[ref_col], infer_datetime_format=True, errors='coerce')
    else:
        df['_ts'] = pd.to_datetime('2000-01-01') + pd.to_timedelta(df.index, unit='s')
        print('  Warning: no timestamp column, using row order.')
    df['timestep'] = pd.cut(df['_ts'].astype('int64'), bins=49, labels=False).fillna(0).astype(int)

    # Column sniffing
    amt_col = next((c for c in df.columns if 'amount' in c.lower() and 'receiv' in c.lower()), None)
    if amt_col is None:
        amt_col = next((c for c in df.columns if 'amount' in c.lower()), df.columns[5])
    src_col = next((c for c in df.columns if c.lower() in ('account','from','sender')), 'Account')
    acct_cols = [c for c in df.columns if 'account' in c.lower()]
    dst_col = acct_cols[1] if len(acct_cols) > 1 else df.columns[3]
    launder_col = next((c for c in df.columns if 'launder' in c.lower()), None)
    print(f'  src={src_col!r} dst={dst_col!r} amt={amt_col!r} launder={launder_col!r}')

    df[src_col] = df[src_col].astype(str)
    df[dst_col] = df[dst_col].astype(str)
    all_accs = pd.unique(pd.concat([df[src_col], df[dst_col]]))
    if len(all_accs) > max_nodes: all_accs = all_accs[:max_nodes]
    acc2idx = {a: i for i, a in enumerate(all_accs)}
    n_nodes = len(all_accs)
    print(f'  Unique accounts (capped): {n_nodes}')

    df_f = df[df[src_col].isin(acc2idx) & df[dst_col].isin(acc2idx)].copy()
    df_f['s_idx'] = df_f[src_col].map(acc2idx)
    df_f['d_idx'] = df_f[dst_col].map(acc2idx)
    df_f[amt_col] = pd.to_numeric(df_f[amt_col], errors='coerce').fillna(0.0)

    print('  Building node features...')
    feats     = np.zeros((n_nodes, 12), dtype=np.float32)
    idx_range = range(n_nodes)
    grp       = df_f.groupby('s_idx')[amt_col]
    feats[:, 0]  = grp.count().reindex(idx_range, fill_value=0).values
    feats[:, 1]  = grp.sum()  .reindex(idx_range, fill_value=0).values
    feats[:, 2]  = grp.max()  .reindex(idx_range, fill_value=0).values
    feats[:, 3]  = grp.std()  .reindex(idx_range, fill_value=0).fillna(0).values
    feats[:, 11] = grp.mean() .reindex(idx_range, fill_value=0).values
    feats[:, 4]  = df_f.groupby('s_idx')['d_idx'].nunique().reindex(idx_range, fill_value=0).values
    feats[:, 5]  = df_f.groupby('d_idx')['s_idx'].nunique().reindex(idx_range, fill_value=0).values
    feats[:, 8]  = df_f.groupby('s_idx')['timestep'].nunique().reindex(idx_range, fill_value=0).values
    df_f['is_round'] = ((df_f[amt_col] % 100) == 0).astype(float)
    feats[:, 10] = df_f.groupby('s_idx')['is_round'].mean().reindex(idx_range, fill_value=0).values
    tc = feats[:, 0] + 1
    feats[:, 6] = feats[:, 4] / tc
    feats[:, 7] = feats[:, 5] / tc
    for s_idx, grp_ts in df_f.groupby('s_idx')['timestep']:
        counts = grp_ts.value_counts(normalize=True).values
        feats[s_idx, 9] = float(-np.sum(counts * np.log(counts + 1e-9)))
    feats = StandardScaler().fit_transform(feats).astype(np.float32)

    labels = np.zeros(n_nodes, dtype=np.int64)
    if launder_col:
        launder_df = df_f[df_f[launder_col] == 1]
        for col in [src_col, dst_col]:
            ill_idx = launder_df[col].map(acc2idx).dropna().astype(int).values
            labels[ill_idx] = 1

    node_ts = np.zeros(n_nodes, dtype=np.int64)
    dom_ts  = df_f.groupby('s_idx')['timestep'].agg(lambda x: x.mode()[0])
    node_ts[dom_ts.index.values] = dom_ts.values.astype(np.int64)

    ei = torch.tensor([df_f['s_idx'].values.tolist(), df_f['d_idx'].values.tolist()], dtype=torch.long)
    data = Data(
        x          = torch.tensor(feats),
        edge_index = ei,
        y          = torch.tensor(labels, dtype=torch.long),
        timestep   = torch.tensor(node_ts, dtype=torch.long)
    )
    print(f'  Nodes: {data.num_nodes} | Edges: {data.num_edges} | Features: {data.num_node_features}')
    print(f'  Illicit: {labels.sum()} | Licit: {(labels==0).sum()}')
    return data

ibm_data = load_ibm_aml()


## Section 3: Federated Partitioning

In [ ]:
def make_mask(n, idx):
    m = torch.zeros(n, dtype=torch.bool)
    if len(idx):
        m[torch.tensor(np.array(idx), dtype=torch.long)] = True
    return m


def temporal_federated_split(data, n_clients=4, test_ratio=0.3):
    # FIX: ONLY labeled nodes (y >= 0) enter train/test masks
    # This is the root cause of -1 labels corrupting weighted_ce
    labels    = data.y.numpy()
    timesteps = data.timestep.numpy()
    valid_idx  = np.where(labels >= 0)[0]
    sorted_idx = valid_idx[np.argsort(timesteps[valid_idx])]
    splits     = np.array_split(sorted_idx, n_clients)

    def strat_split(arr):
        if len(arr) == 0: return arr, arr
        n_te = max(1, int(len(arr) * test_ratio))
        idx  = np.random.permutation(len(arr))
        return arr[idx[n_te:]], arr[idx[:n_te]]

    clients = []
    for i, split in enumerate(splits):
        illicit        = split[labels[split] == 1]
        licit          = split[labels[split] == 0]
        ill_tr, ill_te = strat_split(illicit)
        lic_tr, lic_te = strat_split(licit)
        tr = np.concatenate([ill_tr, lic_tr])
        te = np.concatenate([ill_te, lic_te])
        clients.append({
            'id': i,
            'train_mask': make_mask(data.num_nodes, tr),
            'test_mask':  make_mask(data.num_nodes, te),
            'n_train': len(tr), 'n_test': len(te)
        })
        print(f'  Client {i}: train={len(tr)} test={len(te)} '
              f'ill_train={len(ill_tr)} ill_test={len(ill_te)}')
    return clients


def kfold_federated_split(data, n_clients=4, n_folds=5):
    labels    = data.y.numpy()
    timesteps = data.timestep.numpy()
    valid_idx  = np.where(labels >= 0)[0]
    sorted_idx = valid_idx[np.argsort(timesteps[valid_idx])]
    slices     = np.array_split(sorted_idx, n_clients)
    skf        = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    all_folds  = []
    for fold_idx in range(n_folds):
        fold_clients = []
        for cid, sl in enumerate(slices):
            y_sl = labels[sl]
            if len(np.unique(y_sl)) < 2:
                fold_clients.append({'id': cid,
                    'train_mask': make_mask(data.num_nodes, sl),
                    'test_mask':  make_mask(data.num_nodes, np.array([], dtype=int)),
                    'n_train': len(sl), 'n_test': 0})
                continue
            sp = list(skf.split(sl, y_sl))[fold_idx]
            fold_clients.append({'id': cid,
                'train_mask': make_mask(data.num_nodes, sl[sp[0]]),
                'test_mask':  make_mask(data.num_nodes, sl[sp[1]]),
                'n_train': len(sp[0]), 'n_test': len(sp[1])})
        all_folds.append(fold_clients)
    print(f'  5-fold CV: {n_folds} folds x {n_clients} clients')
    return all_folds


N_CLIENTS = 4
print('=== Elliptic - Temporal Train/Test Split ===')
elliptic_clients = temporal_federated_split(elliptic_data, n_clients=N_CLIENTS)

print('\n=== IBM - 5-Fold Stratified CV Split ===')
ibm_folds   = kfold_federated_split(ibm_data, n_clients=N_CLIENTS, n_folds=5)
ibm_clients = ibm_folds[0]


## Section 4: Model Definitions

In [ ]:
class GATEncoder(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, out_dim=EMB, heads=HEADS, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.conv1   = GATConv(in_dim,         hidden,  heads=heads, dropout=dropout)
        self.conv2   = GATConv(hidden * heads, hidden,  heads=heads, dropout=dropout)
        self.conv3   = GATConv(hidden * heads, out_dim, heads=1,     dropout=dropout, concat=False)
        self.bn1     = nn.BatchNorm1d(hidden * heads)
        self.bn2     = nn.BatchNorm1d(hidden * heads)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv3(x, edge_index)


class ClassHead(nn.Module):
    def __init__(self, in_dim=EMB, hidden=16, n_classes=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x): return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, emb_dim=EMB, n_classes=2,
                 heads=HEADS, dropout=0.6):
        super().__init__()
        self.encoder = GATEncoder(in_dim, hidden, emb_dim, heads, dropout)
        self.head    = ClassHead(emb_dim, 16, n_classes, 0.3)

    def forward(self, x, edge_index):
        emb = self.encoder(x, edge_index)
        return self.head(emb), emb


class SAGEModel(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, out_dim=EMB, n_classes=2, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim,  hidden)
        self.conv2 = SAGEConv(hidden,  hidden)
        self.conv3 = SAGEConv(hidden,  out_dim)
        self.head  = nn.Linear(out_dim, n_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        emb = self.conv3(x, edge_index)
        return self.head(emb), emb

print('Model definitions loaded.')


## Section 5: Utility Functions

In [ ]:
def evaluate(model, data, mask, device):
    # FIX: always restrict evaluation to labeled nodes only
    model.eval()
    with torch.no_grad():
        logits, _ = model(data.x.to(device), data.edge_index.to(device))
    labeled = mask & (data.y >= 0)
    if labeled.sum() == 0:
        return {'acc':0.,'f1':0.,'auc':0.,'prec':0.,'rec':0.,'cm':np.zeros((2,2),dtype=int)}
    preds = logits[labeled].argmax(1).cpu().numpy()
    probs = F.softmax(logits[labeled], dim=1)[:, 1].cpu().numpy()
    true  = data.y[labeled].cpu().numpy()
    return {
        'acc':  accuracy_score(true, preds),
        'f1':   f1_score(true, preds, zero_division=0),
        'auc':  roc_auc_score(true, probs) if len(np.unique(true)) > 1 else 0.0,
        'prec': precision_score(true, preds, zero_division=0),
        'rec':  recall_score(true, preds, zero_division=0),
        'cm':   confusion_matrix(true, preds)
    }


def weighted_ce(logits, labels, device):
    # FIX: normalize by n_classes not sum - preserves illicit class boost
    # Equivalent to sklearn balanced class_weight='balanced'
    n_classes = logits.shape[1]
    counts    = torch.bincount(labels, minlength=n_classes).float()
    weight    = labels.shape[0] / (n_classes * (counts + 1e-8))
    return F.cross_entropy(logits, labels, weight=weight.to(device))


def local_train(model, data, train_mask, device, epochs=5, lr=0.005,
                global_model=None, mu=0.01):
    # FIX: pre-filter -1 labels before loss computation
    # FIX: global_model=None means plain SGD (no proximal term)
    model.train()
    opt   = Adam(model.parameters(), lr=lr)
    x, ei = data.x.to(device), data.edge_index.to(device)
    vmask = train_mask & (data.y >= 0)   # labeled only
    lbls  = data.y[vmask].to(device)
    g_params = ({n: p.detach().clone() for n, p in global_model.named_parameters()}
                if global_model is not None else None)
    for _ in range(epochs):
        opt.zero_grad()
        logits, _ = model(x, ei)
        loss = weighted_ce(logits[vmask], lbls, device)
        if g_params is not None and mu > 0:
            prox = sum(((p - g_params[n])**2).sum() for n, p in model.named_parameters())
            loss = loss + (mu / 2) * prox
        loss.backward()
        opt.step()


def federated_average(global_model, local_models, client_sizes):
    total     = sum(client_sizes)
    g_state   = global_model.state_dict()
    new_state = {}
    for key in g_state:
        acc = torch.zeros_like(g_state[key].float())
        for i, lm in enumerate(local_models):
            acc += lm.state_dict()[key].float() * (client_sizes[i] / total)
        new_state[key] = acc.to(g_state[key].dtype)
    global_model.load_state_dict(new_state)
    return global_model

print('Utility functions loaded.')


## Section 6: PGFCL Core

**Degree-weighted prototype:** $p_c^{(k)} = \\frac{\\sum_{i} d_i z_i}{\\sum_i d_i}$

**InfoNCE + prototype anchor:** $\\mathcal{L} = \\mathcal{L}_{InfoNCE} + \\lambda \\mathcal{L}_{proto}$

In [ ]:
def compute_prototypes(embeddings, labels, edge_index, n_nodes, mask, device,
                       degree_weighted=True):
    """
    Compute per-client, per-class prototype vectors.
    Unchanged from original — the fix is in aggregate_prototypes below.
    """
    labeled = mask & (labels >= 0)
    if degree_weighted:
        row = edge_index[0]
        deg = torch.zeros(n_nodes, dtype=torch.float, device=device)
        deg.scatter_add_(0, row, torch.ones(row.size(0), dtype=torch.float, device=device))
        deg = deg + 1.0   # add-1 smoothing so isolated nodes still contribute
    protos = {}
    for c in [0, 1]:
        cm = labeled & (labels == c)
        if cm.sum() == 0:
            protos[c] = torch.zeros(embeddings.shape[1], device=device)
            continue
        emb_c = embeddings[cm]
        if degree_weighted:
            w = deg[cm].unsqueeze(1)
            protos[c] = (emb_c * w).sum(0) / w.sum()
        else:
            protos[c] = emb_c.mean(0)
    return protos
 
 
# FIX 1 — EMA prototype aggregation
# Old version recomputed the global prototype from scratch every round.
# On non-IID splits this causes the prototype to jump around (you saw licit
# drift climbing to 18 L2 with no sign of convergence).
# EMA smoothing with momentum=0.9 keeps the long-run centroid stable while
# still allowing it to adapt as the encoder improves.
def aggregate_prototypes(client_protos, client_cls_counts,
                         prev_global_protos=None, ema_momentum=0.9):
    """
    Weighted average of client prototypes, then blend with the previous
    global prototype via EMA.
 
    Args:
        client_protos      : list of {0: tensor, 1: tensor} from each client
        client_cls_counts  : list of {0: int, 1: int} — labeled node counts
        prev_global_protos : {0: tensor, 1: tensor} from the previous round,
                             or None for round 1 (no EMA applied then)
        ema_momentum       : weight given to the previous global prototype.
                             0.9 means "keep 90% of old, blend in 10% new".
                             Tune between 0.8–0.95; lower = faster adaptation.
    Returns:
        new_global_protos  : {0: tensor, 1: tensor}
    """
    new_protos = {}
    for c in [0, 1]:
        total = sum(cc.get(c, 0) for cc in client_cls_counts)
        if total == 0:
            new_protos[c] = torch.zeros_like(client_protos[0][c])
            continue
        # Weighted average across clients (same as before)
        weighted_sum = sum(
            client_protos[i][c] * client_cls_counts[i].get(c, 0)
            for i in range(len(client_protos))
        )
        fresh = weighted_sum / total
 
        # EMA blend — only after round 1
        if prev_global_protos is not None and c in prev_global_protos:
            new_protos[c] = ema_momentum * prev_global_protos[c] + (1.0 - ema_momentum) * fresh
        else:
            new_protos[c] = fresh   # round 1: use fresh directly
 
    return new_protos
 
 
# FIX 2 — Detached prototype loss
# Old version: prototype_loss backpropped through z_norm → encoder weights.
# This forced the encoder to move toward the (noisy) global prototype, which
# corrupted the InfoNCE landscape.
# New version: prototype loss uses z_detached — gradients only flow through
# the linear classification head, not the encoder.  The encoder is shaped
# only by InfoNCE (structural self-supervision), while the head is shaped
# by prototype alignment (semantic supervision).  Cleaner separation of roles.
def prototype_anchored_infonce(z, edge_index, labels, mask, global_protos,
                                device, tau=0.5, lam=0.5,
                                feat_drop=0.3, edge_drop=0.2):
    """
    InfoNCE loss (shapes the encoder) + prototype alignment loss (shapes the head).
 
    The prototype term is computed on z.detach() so its gradients do NOT
    propagate through the encoder.  This is the key fix.
    """
    z_norm    = F.normalize(z, dim=1)
    feat_mask = (torch.rand(z.shape[1], device=device) > feat_drop)
    z_aug     = F.normalize(z_norm * feat_mask.float(), dim=1)
 
    src, dst = edge_index
    keep     = torch.rand(src.shape[0], device=device) > edge_drop
    src_k, dst_k = src[keep], dst[keep]
 
    # ── InfoNCE loss (gradients flow into encoder as before) ──────────────
    if src_k.numel() == 0:
        infonce_loss = torch.tensor(0.0, device=device)
    else:
        pos_sim = (z_norm[src_k] * z_aug[dst_k]).sum(1) / tau
        n_neg   = min(512, z_norm.shape[0])
        neg_idx = torch.randperm(z_norm.shape[0], device=device)[:n_neg]
        neg_sim = torch.mm(z_norm[src_k], z_norm[neg_idx].T) / tau
        logits  = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
        tgt     = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
        infonce_loss = F.cross_entropy(logits, tgt)
 
    if global_protos is None or lam == 0.0:
        return infonce_loss
 
    labeled = mask & (labels >= 0)
    if not labeled.any():
        return infonce_loss
 
    # ── Prototype loss — NOTE: z.detach() ─────────────────────────────────
    # We detach z so this loss only propagates into whatever is downstream of
    # z in the optimizer graph.  During ssl_pretrain_round the encoder IS the
    # only thing in the graph, so we explicitly stop gradients here.
    # During supervised fine-tuning (local_train) the head is also in the
    # graph, so the head still learns from prototype alignment.
    z_detached   = z.detach()                      # <── THE FIX
    z_lab        = F.normalize(z_detached[labeled], dim=1)
    y_lab        = labels[labeled]
    p0           = F.normalize(global_protos[0].unsqueeze(0), dim=1)
    p1           = F.normalize(global_protos[1].unsqueeze(0), dim=1)
    protos_cat   = torch.cat([p0, p1], dim=0)
    proto_logits = torch.mm(z_lab, protos_cat.T) / tau
    proto_loss   = F.cross_entropy(proto_logits, y_lab)
 
    return infonce_loss + lam * proto_loss
 
 
def ssl_pretrain_round(encoder, data, mask, device, global_protos=None,
                       epochs=20, lr=0.005, tau=0.5, lam=0.5):
    """Unchanged API — the fix is inside prototype_anchored_infonce."""
    encoder.train()
    opt    = Adam(encoder.parameters(), lr=lr)
    x, ei  = data.x.to(device), data.edge_index.to(device)
    y      = data.y.to(device)
    for _ in range(epochs):
        opt.zero_grad()
        z    = encoder(x, ei)
        loss = prototype_anchored_infonce(
            z, ei, y, mask.to(device), global_protos, device,
            tau=tau, lam=lam if global_protos is not None else 0.0
        )
        loss.backward()
        opt.step()

## Section 7: Baseline 1 - Centralized GAT

In [ ]:
def run_centralized_gat(data, clients, device, epochs=100, lr=0.005):
    print('\n[Centralized GAT] Training...')
    model = FullGAT(data.num_node_features).to(device)
    opt   = Adam(model.parameters(), lr=lr)
    all_train = torch.zeros(data.num_nodes, dtype=torch.bool)
    all_test  = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients:
        all_train |= c['train_mask']
        all_test  |= c['test_mask']
    # FIX: pre-filter -1 labels
    vmask  = all_train & (data.y >= 0)
    labels = data.y[vmask].to(device)
    x, ei  = data.x.to(device), data.edge_index.to(device)
    best_f1, best_m = 0, {}
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        logits, _ = model(x, ei)
        weighted_ce(logits[vmask], labels, device).backward()
        opt.step()
        if (ep + 1) % 20 == 0:
            m = evaluate(model, data, all_test, device)
            if m['f1'] > best_f1: best_f1, best_m = m['f1'], m
            print(f'  Ep {ep+1:3d} | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    print(f'  Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    return best_m

central_elliptic = run_centralized_gat(elliptic_data, elliptic_clients, DEVICE)


## Section 8: Baseline 2 - Local-Only GAT

In [ ]:
def run_local_only_gat(data, clients, device, epochs=50, lr=0.005):
    print('\n[Local-Only GAT] Training...')
    x, ei = data.x.to(device), data.edge_index.to(device)
    all_metrics = []
    for client in clients:
        model = FullGAT(data.num_node_features).to(device)
        local_train(model, data, client['train_mask'], device, epochs=epochs, lr=lr)
        m = evaluate(model, data, client['test_mask'], device)
        all_metrics.append(m)
        print(f'  Client {client["id"]} | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    avg = {k: float(np.mean([m[k] for m in all_metrics]))
           for k in ['acc','f1','auc','prec','rec']}
    print(f'  Avg F1={avg["f1"]:.4f} | Avg AUC={avg["auc"]:.4f}')
    return avg

local_elliptic = run_local_only_gat(elliptic_data, elliptic_clients, DEVICE)


## Section 9: Baseline 3 - FedAvg GAT

In [ ]:
def run_fedavg_gat(data, clients, device, global_rounds=10, local_epochs=5, lr=0.005):
    print('\n[FedAvg GAT] Training...')
    global_model    = FullGAT(data.num_node_features).to(device)
    best_f1, best_m = 0, {}
    all_test = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients: all_test |= c['test_mask']
    for rnd in range(global_rounds):
        local_models = []
        for client in clients:
            lm = FullGAT(data.num_node_features).to(device)
            lm.load_state_dict(global_model.state_dict())
            # FIX: plain local_train with mu=0 (no unnecessary proximal overhead)
            local_train(lm, data, client['train_mask'], device,
                        epochs=local_epochs, lr=lr)
            local_models.append(lm)
        sizes        = [c['n_train'] for c in clients]
        global_model = federated_average(global_model, local_models, sizes)
        m = evaluate(global_model, data, all_test, device)
        if m['f1'] > best_f1: best_f1, best_m = m['f1'], m
        print(f'  Round {rnd+1:2d} | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    print(f'  Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    return best_m

fedavg_elliptic = run_fedavg_gat(elliptic_data, elliptic_clients, DEVICE)


## Section 10: Baseline 4 - FedSage

In [ ]:
def run_fedsage(data, clients, device, global_rounds=10, local_epochs=5, lr=0.005):
    print('\n[FedSage] Training...')
    global_model    = SAGEModel(data.num_node_features).to(device)
    best_f1, best_m = 0, {}
    all_test = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients: all_test |= c['test_mask']
    x, ei = data.x.to(device), data.edge_index.to(device)
    for rnd in range(global_rounds):
        local_models = []
        for client in clients:
            lm = SAGEModel(data.num_node_features).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt   = Adam(lm.parameters(), lr=lr)
            # FIX: filter -1 labels
            vmask = client['train_mask'] & (data.y >= 0)
            lbls  = data.y[vmask].to(device)
            for _ in range(local_epochs):
                opt.zero_grad()
                logits, _ = lm(x, ei)
                weighted_ce(logits[vmask], lbls, device).backward()
                opt.step()
            local_models.append(lm)
        sizes        = [c['n_train'] for c in clients]
        global_model = federated_average(global_model, local_models, sizes)
        m = evaluate(global_model, data, all_test, device)
        if m['f1'] > best_f1: best_f1, best_m = m['f1'], m
        print(f'  Round {rnd+1:2d} | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    print(f'  Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    return best_m

fedsage_elliptic = run_fedsage(elliptic_data, elliptic_clients, DEVICE)


## Section 11: PGFCL - Full System

In [ ]:
def run_pgfcl(data, clients, device,
              global_rounds=20, ssl_epochs=20, sup_epochs=15,
              lr=0.005, mu=0.01, tau=0.5,
              lam_max=0.5, lam_warmup_rounds=5,
              ema_momentum=0.9,               # new — controls prototype EMA
              use_prototypes=True, use_degree_weighting='auto',
              verbose=True):
    """
    Full PGFCL training loop with EMA prototype stabilisation.
 
    New parameter:
        ema_momentum (float): EMA weight for prototype smoothing.
            0.9 is a safe default.  Try 0.8 if prototypes converge too slowly,
            0.95 if drift is still noisy after the fix.
    """
    use_dw = (data.num_edges > 100_000) if use_degree_weighting == 'auto' \
              else bool(use_degree_weighting)
 
    global_model    = FullGAT(data.num_node_features).to(device)
    global_protos   = None    # None until after round 1
    prev_protos     = None    # for EMA and drift tracking
    drift_curve     = []
    best_f1, best_m = 0, {}
 
    all_test = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients:
        all_test |= c['test_mask']
    x, ei = data.x.to(device), data.edge_index.to(device)
 
    for rnd in range(global_rounds):
        lam = lam_max * min(1.0, rnd / max(lam_warmup_rounds, 1))
        if verbose:
            print(f'\n  === Round {rnd+1}/{global_rounds} | lam={lam:.3f} ===')
 
        # ── Phase 1: SSL pretraining per client ───────────────────────────
        client_encoders    = []
        client_protos_list = []
        client_cls_counts  = []
 
        for client in clients:
            encoder = GATEncoder(data.num_node_features).to(device)
            encoder.load_state_dict(global_model.encoder.state_dict())
            ssl_pretrain_round(
                encoder, data, client['train_mask'], device,
                global_protos=global_protos if use_prototypes else None,
                epochs=ssl_epochs, lr=lr, tau=tau, lam=lam
            )
            encoder.eval()
            with torch.no_grad():
                z = encoder(x, ei)
 
            tr_mask = client['train_mask']
            protos  = compute_prototypes(
                z, data.y.to(device), ei, data.num_nodes,
                tr_mask.to(device), device, degree_weighted=use_dw
            )
            cls_counts = {c_id: int((tr_mask & (data.y == c_id)).sum())
                          for c_id in [0, 1]}
            client_encoders.append(encoder)
            client_protos_list.append(protos)
            client_cls_counts.append(cls_counts)
 
        # ── Phase 2: EMA prototype aggregation ────────────────────────────
        # FIX 1 applied here — pass prev_protos for EMA blending
        global_protos = aggregate_prototypes(
            client_protos_list, client_cls_counts,
            prev_global_protos=prev_protos,          # <── EMA FIX
            ema_momentum=ema_momentum if use_prototypes else 0.0
        )
 
        # Drift tracking (L2 between consecutive global prototypes)
        if prev_protos is not None:
            d0 = (global_protos[0] - prev_protos[0]).norm().item()
            d1 = (global_protos[1] - prev_protos[1]).norm().item()
            drift_curve.append({'round': rnd+1, 'drift_licit': d0, 'drift_illicit': d1})
            if verbose:
                print(f'  Prototype drift - licit: {d0:.4f} | illicit: {d1:.4f}')
        prev_protos = {c: v.detach().clone() for c, v in global_protos.items()}
 
        # ── Phase 3: FedProx supervised fine-tuning ───────────────────────
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.encoder.load_state_dict(client_encoders[i].state_dict())
            local_train(lm, data, client['train_mask'], device,
                        epochs=sup_epochs, lr=lr,
                        global_model=global_model, mu=mu)
            local_models.append(lm)
 
        # ── Phase 4: FedAvg ───────────────────────────────────────────────
        sizes        = [c['n_train'] for c in clients]
        global_model = federated_average(global_model, local_models, sizes)
 
        m = evaluate(global_model, data, all_test, device)
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m
        if verbose:
            print(f'  Global | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f} | '
                  f'Prec={m["prec"]:.4f} | Rec={m["rec"]:.4f}')
 
    print(f'\n[PGFCL] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    return best_m, drift_curve
 
 
# Helper functions below are unchanged from your original — included here
# for completeness so this file is self-contained as a reference.
 
def run_pgfcl_kfold(data, all_folds, device, n_folds=5, **kwargs):
    fold_metrics = []
    for fold_idx, clients in enumerate(all_folds):
        print(f'\n  --- Fold {fold_idx+1}/{n_folds} ---')
        m, _ = run_pgfcl(data, clients, device, verbose=False, **kwargs)
        fold_metrics.append(m)
        print(f'    F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    keys = ['acc', 'f1', 'auc', 'prec', 'rec']
    avg  = {k: float(np.mean([m[k] for m in fold_metrics])) for k in keys}
    std  = {k: float(np.std( [m[k] for m in fold_metrics])) for k in keys}
    print(f'\n[PGFCL 5-CV] F1={avg["f1"]:.4f}+-{std["f1"]:.4f} | '
          f'AUC={avg["auc"]:.4f}+-{std["auc"]:.4f}')
    avg['std'] = std
    avg['fold_metrics'] = fold_metrics
    return avg
 
 
def run_baseline_kfold(run_fn, data, all_folds, device, n_folds=5, **kwargs):
    fold_metrics = []
    for fold_idx, clients in enumerate(all_folds):
        print(f'  Fold {fold_idx+1}/{n_folds}', end=' ')
        m = run_fn(data, clients, device, **kwargs)
        if isinstance(m, tuple):
            m = m[0]
        fold_metrics.append(m)
        print(f'F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    keys = ['acc', 'f1', 'auc', 'prec', 'rec']
    avg  = {k: float(np.mean([m[k] for m in fold_metrics])) for k in keys}
    std  = {k: float(np.std( [m[k] for m in fold_metrics])) for k in keys}
    avg['std'] = std
    return avg
 

## Section 11b: IBM Baselines - 5-Fold CV

In [ ]:
print('\n=== IBM Baselines - 5-fold CV ===')
central_ibm = run_baseline_kfold(run_centralized_gat, ibm_data, ibm_folds, DEVICE, epochs=100)
local_ibm   = run_baseline_kfold(run_local_only_gat,  ibm_data, ibm_folds, DEVICE, epochs=50)
fedavg_ibm  = run_baseline_kfold(run_fedavg_gat,      ibm_data, ibm_folds, DEVICE,
                                  global_rounds=10, local_epochs=5)
fedsage_ibm = run_baseline_kfold(run_fedsage,          ibm_data, ibm_folds, DEVICE,
                                  global_rounds=10, local_epochs=5)


## Section 12: Ablation Study

In [ ]:
def run_ablation(data, clients_or_folds, device, dataset_name='Elliptic', use_cv=False):
    print(f'\n===== Ablation Study - {dataset_name} =====')
    results = {}
    configs = [
        ('Full PGFCL',               True,  'auto'),
        ('No degree-weighting',      True,  False),
        ('No prototype feedback',    False, 'auto'),
        ('SSL only (no prototypes)', False, 'auto'),
    ]
    # FIX: lam_max NOT in shared base_kw - set per config to avoid duplicate key error
    base_kw = dict(global_rounds=15, ssl_epochs=20, sup_epochs=12, lam_warmup_rounds=5)
    for name, use_proto, use_dw in configs:
        print(f'  Config: {name}')
        kw = dict(**base_kw,
                  use_prototypes=use_proto,
                  use_degree_weighting=use_dw,
                  lam_max=0.5 if use_proto else 0.0)
        if use_cv:
            m = run_baseline_kfold(
                lambda d, c, dev, **k: run_pgfcl(d, c, dev, verbose=False, **k)[0],
                data, clients_or_folds, device, **kw)
        else:
            m, _ = run_pgfcl(data, clients_or_folds, device, verbose=False, **kw)
        results[name] = m
        std = m.get('std', {})
        sfx = f'+-{std["f1"]:.4f}' if std else ''
        print(f'    F1={m["f1"]:.4f}{sfx} | AUC={m["auc"]:.4f}')
    return results

ablation_elliptic = run_ablation(elliptic_data, elliptic_clients, DEVICE, 'Elliptic')
ablation_ibm      = run_ablation(ibm_data, ibm_folds, DEVICE, 'IBM', use_cv=True)


## Section 13: Lambda Sensitivity Analysis

In [ ]:
def lambda_sensitivity(data, clients_or_folds, device, dataset_name='Elliptic',
                       use_cv=False, lam_values=None):
    if lam_values is None:
        lam_values = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
    print(f'\n===== Lambda Sensitivity - {dataset_name} =====')
    f1s, aucs = [], []
    base_kw = dict(global_rounds=15, ssl_epochs=20, sup_epochs=12, lam_warmup_rounds=4)
    for lam in lam_values:
        if use_cv:
            m = run_baseline_kfold(
                lambda d, c, dev, **k: run_pgfcl(d, c, dev, verbose=False, **k)[0],
                data, clients_or_folds, device, lam_max=lam, **base_kw)
        else:
            m, _ = run_pgfcl(data, clients_or_folds, device,
                             lam_max=lam, verbose=False, **base_kw)
        f1s.append(m['f1']); aucs.append(m['auc'])
        sfx = f'+-{m["std"]["f1"]:.4f}' if m.get('std') else ''
        print(f'  lam={lam:.2f} | F1={m["f1"]:.4f}{sfx} | AUC={m["auc"]:.4f}')
    return lam_values, f1s, aucs

lams_e, f1s_e, aucs_e = lambda_sensitivity(elliptic_data, elliptic_clients, DEVICE, 'Elliptic')
lams_i, f1s_i, aucs_i = lambda_sensitivity(ibm_data, ibm_folds, DEVICE, 'IBM', use_cv=True)


## Section 14: Visualization

In [ ]:
def plot_results_table():
    models    = ['Centralized GAT','Local-Only GAT','FedAvg GAT','FedSage','PGFCL (Ours)']
    e_metrics = [central_elliptic, local_elliptic, fedavg_elliptic, fedsage_elliptic, pgfcl_elliptic]
    i_metrics = [central_ibm,      local_ibm,      fedavg_ibm,     fedsage_ibm,      pgfcl_ibm]
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    keys    = ['acc','f1','auc','prec','rec']
    headers = ['Model','Acc','F1','AUC','Prec','Rec']
    for ax, metrics, title in zip(axes, [e_metrics, i_metrics], ['Elliptic','IBM AML']):
        cell_text = [[mod] + [f'{m.get(k,0):.4f}' for k in keys]
                     for mod, m in zip(models, metrics)]
        ax.axis('off')
        t = ax.table(cellText=cell_text, colLabels=headers, cellLoc='center', loc='center')
        t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.5)
        for j in range(len(headers)):
            t[len(models), j].set_facecolor('#d4edda')
        ax.set_title(title, fontsize=12, fontweight='bold', pad=20)
    plt.suptitle('PGFCL vs Baselines - Performance Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results_table.pdf', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: results_table.pdf')

plot_results_table()


In [ ]:
def plot_prototype_drift(drift_e, drift_i):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, drift, title in zip(axes, [drift_e, drift_i], ['Elliptic','IBM AML']):
        if not drift:
            ax.text(0.5, 0.5, 'No drift data', ha='center', va='center')
            ax.set_title(title, fontsize=12, fontweight='bold'); continue
        rounds = [d['round'] for d in drift]
        d_lic  = [d['drift_licit']   for d in drift]
        d_ill  = [d['drift_illicit'] for d in drift]
        ax.plot(rounds, d_ill, 'o-',  color='#c0392b', lw=2, ms=7, label='Illicit drift')
        ax.plot(rounds, d_lic, 's--', color='#2980b9', lw=2, ms=7, label='Licit drift')
        pk = int(np.argmax(d_ill))
        ax.annotate(f'Max\n(Rnd {rounds[pk]})', xy=(rounds[pk], d_ill[pk]),
                    xytext=(rounds[pk]+0.5, d_ill[pk]*0.85),
                    arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9, color='#c0392b')
        ax.fill_between(rounds, d_ill, alpha=0.1, color='#c0392b')
        ax.set_xlabel('Round'); ax.set_ylabel('Prototype Drift (L2)')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(); ax.grid(True, alpha=0.3); ax.set_xticks(rounds)
    plt.suptitle('PGFCL Prototype Drift - AML Audit Signal', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('prototype_drift.pdf', bbox_inches='tight', dpi=150)
    plt.show()

plot_prototype_drift(drift_elliptic, drift_ibm)


In [ ]:
def plot_ablation(abl_e, abl_i):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#2ecc71','#3498db','#e67e22','#e74c3c']
    for ax, ablation, title in zip(axes, [abl_e, abl_i], ['Elliptic','IBM AML']):
        configs = list(ablation.keys())
        f1s  = [ablation[c]['f1']  for c in configs]
        aucs = [ablation[c]['auc'] for c in configs]
        x = np.arange(len(configs)); w = 0.35
        b1 = ax.bar(x - w/2, f1s,  w, label='F1',  color=colors, alpha=0.7)
        b2 = ax.bar(x + w/2, aucs, w, label='AUC', color=colors, alpha=0.9)
        for bar in [*b1, *b2]:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
        # FIX: dynamic ylim instead of hardcoded 0.85 (crashes when F1 < 0.85)
        all_vals = f1s + aucs
        ax.set_ylim(max(0, min(all_vals) - 0.05), min(1.05, max(all_vals) + 0.05))
        ax.set_xticks(x)
        ax.set_xticklabels(configs, rotation=15, ha='right', fontsize=9)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(); ax.grid(True, axis='y', alpha=0.3); ax.set_ylabel('Score')
    plt.suptitle('Ablation Study - PGFCL Component Contributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ablation.pdf', bbox_inches='tight', dpi=150)
    plt.show()

plot_ablation(ablation_elliptic, ablation_ibm)


In [ ]:
def plot_lambda_sensitivity(lams_e, f1s_e, aucs_e, lams_i, f1s_i, aucs_i):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, lams, f1s, aucs, title in zip(
        axes, [lams_e, lams_i], [f1s_e, f1s_i], [aucs_e, aucs_i], ['Elliptic','IBM AML']
    ):
        ax.plot(lams, f1s,  'o-',  color='#2ecc71', lw=2.5, ms=8, label='F1')
        ax.plot(lams, aucs, 's--', color='#3498db', lw=2.5, ms=8, label='AUC')
        best = lams[int(np.argmax(f1s))]
        ax.axvline(x=best, color='gray', ls=':', alpha=0.7)
        ax.text(best + 0.05, min(f1s), f'lam*={best}', fontsize=9, color='gray')
        ax.set_xlabel('lambda (prototype anchor weight)'); ax.set_ylabel('Score')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle('Lambda Sensitivity Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('lambda_sensitivity.pdf', bbox_inches='tight', dpi=150)
    plt.show()

plot_lambda_sensitivity(lams_e, f1s_e, aucs_e, lams_i, f1s_i, aucs_i)


In [ ]:
def plot_confusion_matrices(results_dict, dataset_name):
    models = [k for k in results_dict if 'cm' in results_dict[k]]
    if not models: print('No confusion matrix data.'); return
    n = len(models)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, mod in zip(axes, models):
        sns.heatmap(results_dict[mod]['cm'], annot=True, fmt='d', ax=ax,
                    cmap='Blues', cbar=False,
                    xticklabels=['Licit','Illicit'], yticklabels=['Licit','Illicit'])
        ax.set_title(mod, fontsize=10, fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.suptitle(f'Confusion Matrices - {dataset_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'confusion_{dataset_name.lower()}.pdf', bbox_inches='tight', dpi=150)
    plt.show()

plot_confusion_matrices({
    'Centralized GAT': central_elliptic,
    'FedAvg':          fedavg_elliptic,
    'FedSage':         fedsage_elliptic,
    'PGFCL (Ours)':    pgfcl_elliptic,
}, 'Elliptic')


## Section 15: Final Summary

In [ ]:
print('\n' + '='*72)
print('FINAL RESULTS SUMMARY')
print('Elliptic: train/test | IBM: 5-fold CV mean +- std')
print('='*72)

def fmt(m, key):
    val = m.get(key, 0)
    std = m.get('std', {}).get(key, None)
    return f'{val:.4f}+-{std:.4f}' if std is not None else f'{val:.4f}'

for ds_name, results in [
    ('Elliptic (train/test)', {
        'Centralized GAT': central_elliptic,
        'Local-Only GAT':  local_elliptic,
        'FedAvg GAT':      fedavg_elliptic,
        'FedSage':         fedsage_elliptic,
        'PGFCL (Ours)':    pgfcl_elliptic,
    }),
    ('IBM AML (5-fold CV)', {
        'Centralized GAT': central_ibm,
        'Local-Only GAT':  local_ibm,
        'FedAvg GAT':      fedavg_ibm,
        'FedSage':         fedsage_ibm,
        'PGFCL (Ours)':    pgfcl_ibm,
    }),
]:
    print(f'\n{ds_name}')
    print(f'  {"Model":<22} {"F1":>16} {"AUC":>16} {"Prec":>16} {"Rec":>16}')
    print(f'  {"-"*74}')
    for mod, m in results.items():
        marker = ' <-- PGFCL' if 'PGFCL' in mod else ''
        print(f'  {mod:<22} {fmt(m,"f1"):>16} {fmt(m,"auc"):>16} '
              f'{fmt(m,"prec"):>16} {fmt(m,"rec"):>16}{marker}')

print('\n' + '='*72)
